In [1]:
import pandas as pd
import numpy as np

print("Environment Ready")

Environment Ready


In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Rows, Columns:", df.shape)
df.head()

Rows, Columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df["Churn"].value_counts()

No     5174
Yes    1869
Name: Churn, dtype: int64

In [5]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
df[df["TotalCharges"] == " "].shape

(11, 21)

In [7]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

In [8]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [9]:
df = df.drop("customerID", axis=1)

In [10]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df["Churn"].value_counts()

0    5174
1    1869
Name: Churn, dtype: int64

In [11]:
df = pd.get_dummies(df, drop_first=True)

In [12]:
print(df.shape)

(7043, 31)


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(5634, 30)
(1409, 30)


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7863733144073811


In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.77      0.79      0.78      1409



In [16]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[924 111]
 [190 184]]


In [17]:
import joblib

joblib.dump(model, "churn_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [18]:
import mlflow

print(mlflow.__version__)

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


3.8.1


In [19]:
import os

print(os.path.exists("churn_model.pkl"))

True


In [20]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print("Connected Successfully")

Found the config file in: /config.json


Connected Successfully


In [21]:
from azure.ai.ml.entities import Model

model = Model(
    path="churn_model.pkl",
    name="churn-randomforest-model",
    description="Telco Customer Churn Prediction Model",
    type="custom_model"
)

registered_model = ml_client.models.create_or_update(model)

print(
    f"Model registered: {registered_model.name}, Version: {registered_model.version}"
)

Uploading churn_model.pkl (< 1 MB): 100%|██████████| 19.0M/19.0M [00:00<00:00, 68.6MB/s]




Model registered: churn-randomforest-model, Version: 1


In [23]:
%run src/train.py

Accuracy: 0.7778566359119943
Pipeline model saved successfully


In [24]:
from azure.ai.ml.entities import Model

model = Model(
    path="churn_model.pkl",
    name="churn-randomforest-model",
    description="Pipeline-based churn prediction model",
    type="custom_model"
)

registered_model = ml_client.models.create_or_update(model)

print(
    f"Model registered: {registered_model.name}, Version: {registered_model.version}"
)

Uploading churn_model.pkl (< 1 MB): 0.00B [00:00, ?B/s]

Model registered: churn-randomforest-model, Version: 2


In [26]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name="churn-endpoint",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

print("Endpoint Created Successfully")

HttpResponseError: (SubscriptionNotRegistered) Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider
Code: SubscriptionNotRegistered
Message: Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider

In [27]:
import azure.ai.ml

print(azure.ai.ml.__version__)

1.31.0


In [28]:
print(ml_client.workspaces.get(ml_client.workspace_name))

allow_roleassignment_on_rg: true
application_insights: /subscriptions/e34b4152-e092-4bfd-8fe6-15c7e6d5f7fd/resourceGroups/rg-churn-ml-lab/providers/Microsoft.insights/components/amlchurnlab8561097249
container_registry: /subscriptions/e34b4152-e092-4bfd-8fe6-15c7e6d5f7fd/resourceGroups/rg-churn-ml-lab/providers/Microsoft.ContainerRegistry/registries/amlmllab
description: ''
discovery_url: https://eastasia.api.azureml.ms/discovery
display_name: aml-churn-lab
enable_data_isolation: false
hbi_workspace: false
id: /subscriptions/e34b4152-e092-4bfd-8fe6-15c7e6d5f7fd/resourceGroups/rg-churn-ml-lab/providers/Microsoft.MachineLearningServices/workspaces/aml-churn-lab
identity:
  principal_id: b91ec071-45e4-4891-ac79-e478566a8976
  tenant_id: 8d1cf962-6920-472d-a1fb-16af1f8b5bb5
  type: system_assigned
key_vault: /subscriptions/e34b4152-e092-4bfd-8fe6-15c7e6d5f7fd/resourceGroups/rg-churn-ml-lab/providers/Microsoft.Keyvault/vaults/amlchurnlab1604700016
location: eastasia
managed_network:
  isola

In [29]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name="endpoint-test-123",
    auth_mode="key"
)

poller = ml_client.online_endpoints.begin_create_or_update(endpoint)

poller.result()

HttpResponseError: (SubscriptionNotRegistered) Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider
Code: SubscriptionNotRegistered
Message: Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider

In [30]:
import subprocess

result = subprocess.run(
    ["az", "provider", "list", "--query", "[?registrationState!='Registered'].namespace"],
    capture_output=True,
    text=True
)

print(result.stdout)

In [31]:
!az provider list --query "[?registrationState!='Registered'].namespace"

Please run 'az login' to setup account.


In [32]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name="testendpoint123"
)

print(endpoint)

auth_mode: key
mirror_traffic: {}
name: testendpoint123
properties: {}
tags: {}
traffic: {}



In [33]:
print(ml_client.workspaces.get(ml_client.workspace_name).location)

eastasia
